# Thành viên 1 — Kaggle `GPU T4 x 2` (32 GB VRAM): Track Tiếng Việt (`ViSoBERT-ViHSD + mDeBERTa Stance Verbalizer -> ViSoBERT CE + SpanPool -> SBi-FAGAT`)

**Cấu hình bắt buộc trên Kaggle trước khi chạy:**
1. Bật **Accelerator: GPU T4 x 2** (tổng 32 GB VRAM).
2. Bật **Internet: ON** để kéo repo GitHub, dữ liệu Arctic Shift và trọng số Hugging Face.
3. Bật **Persistence: Files only** (khuyên dùng) để giữ checkpoint nếu ngắt phiên.

Toàn bộ kết quả thực nghiệm được xuất sạch vào **`results/`** (tuyệt đối không ghi đè `paper/`).

Bạn có 2 chế độ chọn nhanh:
- **`member1_vn`** *(Khuyên dùng để chia 50/50 với Bạn 2)*: Chạy trọn gói Tiếng Việt (thu thập 1.500 cây Voz/VietNam -> tinh chỉnh `uitnlp/visobert` trên `UIT-ViHSD` + chấm `mDeBERTa-v3-xnli` Stance Verbalizer với tùy chọn `HostilityGatedTargetRouter` -> chưng cất Cross-Encoder 2D `uitnlp/visobert` -> trích xuất `.pt` *Ancestor SpanPool* -> 5-seed công bằng 15+ mô hình + `SBiFAGAT + Tier2Prior` trên `canonical` + `embedding_cluster` + `temporal` + Ablation + `--include-e2e` Efficiency + tùy chọn Tier 2 VN 10 mô hình).
- **`model_nlp_only`**: Nếu muốn 1 người chuyên train toàn bộ **Mô hình NLP (`ViSoBERT-ViHSD + mDeBERTa Verbalizer + Cross-Encoder 2D`)** (hoặc bật `DISTILL_FROM_FEATURES = True` để khôi phục nhanh `dual_ce_vn/dual_head_weights.pt` từ `data/processed/vn/nlp_features.pt`) rồi gửi sang cho Bạn 2 train GNN.

In [ ]:
# Bước 1: Kiểm tra GPU NVIDIA Tesla T4
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available(), "| Num GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
%%bash
set -euo pipefail
# Bước 2: Clone hoặc cập nhật mã nguồn mới nhất từ GitHub & cài thư viện (khóa đúng torch CUDA của Kaggle)
BRANCH="${SBIF_BRANCH:-main}"
cd /kaggle/working
if [ ! -d "gnn-nlp-propagation/.git" ]; then
  rm -rf gnn-nlp-propagation
  git clone https://github.com/hieu-kiien/gnn-nlp-propagation.git
fi
cd /kaggle/working/gnn-nlp-propagation
git fetch origin
git checkout "$BRANCH"
git pull --ff-only origin "$BRANCH"
python -c 'import torch; print("torch==" + torch.__version__.split("+")[0])' > /tmp/kaggle_constraints.txt
grep -v '^torch>=' requirements.txt > /tmp/req_kaggle.txt
pip install -q -c /tmp/kaggle_constraints.txt -r /tmp/req_kaggle.txt
pytest -q --ignore=tests/e2e

In [ ]:
# Khởi tạo muối băm SHA-256 bền vững (nếu chưa đặt SBIF_HASH_SALT) để giả danh hóa đồng nhất kể cả khi resume
import os, pathlib, secrets
os.chdir("/kaggle/working/gnn-nlp-propagation")
salt_file = pathlib.Path("data/interim/.author_hash_salt.secret")
if not os.environ.get("SBIF_HASH_SALT") and not salt_file.exists():
    salt_file.parent.mkdir(parents=True, exist_ok=True)
    salt_file.write_text(secrets.token_hex(16), encoding="utf-8")
    print("Created persistent salt at", salt_file)

# (Tùy chọn) Đăng nhập Hugging Face nếu repo model yêu cầu token đọc
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# Bước 3: Chạy thử kiểm tra nhanh hệ thống (chạy trong <10 giây để chắc chắn 0 lỗi môi trường)
!python -m scripts.kaggle_parallel_runner --role member1_vn --smoke-test
!python -m scripts.predict_live_thread --demo-offline

In [ ]:
# Bước 4: CHẠY CHÍNH THỨC TRÊN KAGGLE
# Đổi ROLE = "model_nlp_only" nếu bạn chỉ muốn chuyên trách xuất bộ NLP Tiếng Việt (.pt) cho bạn còn lại
import pathlib, shlex, shutil, subprocess, sys
ROLE = "member1_vn"  # hoặc "model_nlp_only"
VN_TARGET_COUNT = 1500
RESULTS_DIR = "results"
INCLUDE_HEAVY_WEIGHTS = True
DISTILL_FROM_FEATURES = False  # Bật True nếu đã mount sẵn data/processed/vn/nlp_features.pt
USE_HOSTILITY_ROUTER = True    # Bật HostilityGatedTargetRouter khi chấm Teacher
INCLUDE_TIER2_PRIOR = True
RUN_EMBEDDING_CLUSTER = True
RUN_TIER2_MACRO = True
MP_GRAPH_MODE = "coalesced"

flags = [
    f"--role {ROLE}",
    "--lang vi",
    f"--vn-target-count {VN_TARGET_COUNT}",
    "--before-epoch 1763164800",
    "--hidden-dim 64",
    "--epochs-nlp 3",
    "--epochs-gnn 100",
    f"--results-dir {RESULTS_DIR}",
]
if INCLUDE_HEAVY_WEIGHTS:
    flags.append("--include-heavy-weights")
if DISTILL_FROM_FEATURES:
    flags.append("--distill-from-features")
if USE_HOSTILITY_ROUTER:
    flags.append("--hostility-target-router")
if INCLUDE_TIER2_PRIOR:
    flags.append("--include-tier2-prior")
if RUN_EMBEDDING_CLUSTER:
    flags.append("--run-embedding-cluster")
if RUN_TIER2_MACRO:
    flags.append(f"--run-tier2 --mp-graph-mode {MP_GRAPH_MODE}")

cmd = "-m scripts.kaggle_parallel_runner " + " ".join(flags)
subprocess.run([sys.executable, "-u", *shlex.split(cmd)], check=True)

working_root = pathlib.Path("/kaggle/working")
if working_root.exists():
    for zname in (f"sbifagat_{ROLE}_bundle.zip", f"sbifagat_{ROLE}_sealed_machine_keys.zip"):
        zpath = pathlib.Path(zname)
        if zpath.is_file() and zpath.resolve().parent != working_root.resolve():
            shutil.copy2(zpath, working_root / zpath.name)
            print("Mirrored zip to", working_root / zpath.name)

In [ ]:
# Bước 5: Chạy kiểm thử suy luận offline & tóm tắt E2E latency trên checkpoint vừa train
import json, pathlib, shlex, subprocess, sys
if pathlib.Path("checkpoints/best_sbifagat_vn.pt").is_file():
    subprocess.run(
        [sys.executable, *shlex.split("-m scripts.predict_live_thread --demo-offline --cross-encoder-ckpt checkpoints/dual_ce_vn --gnn-ckpt checkpoints/best_sbifagat_vn.pt")],
        check=True,
    )
else:
    print("[SKIP] Chưa có checkpoints/best_sbifagat_vn.pt (bỏ qua khi chạy ROLE='model_nlp_only').")

eff_p = pathlib.Path("results/results_efficiency_vn.json")
if eff_p.is_file():
    eff = json.loads(eff_p.read_text(encoding="utf-8"))
    print("E2E latency summary:", json.dumps(eff.get("e2e_raw_text_profile", {}), indent=2))

## Xuất kết quả & bảo toàn đáp án chấm mù
Sau khi ô trên chạy xong, tải đồng thời **2 gói nén** trong `/kaggle/working/` (hoặc `/kaggle/working/gnn-nlp-propagation/`) trước khi tắt máy ảo Kaggle:
1. `sbifagat_member1_vn_bundle.zip` *(hoặc `sbifagat_model_nlp_only_bundle.zip`)* — chứa kết quả chính `results/results_main_vn.json`, `results/results_cluster_vn.json`, `results/results_temporal_vn.json`, `results/results_ablation_vn.json`, `results/results_efficiency_vn.json`, `results/results_macro_network_vn.json`, `results/results_signed_link_vn.json`, `results/tables_*_vn.tex`, `results/figures/vn/*.pdf`, `checkpoints/best_sbifagat_vn.pt`, trọng số NLP (`checkpoints/visobert_vihsd/*`, `checkpoints/dual_ce_vn/dual_head_weights.pt`, `data/processed/vn/nlp_features.pt` khi bật `--include-heavy-weights`), phiếu pilot `data/interim/pilot_calibration_vi_20.csv`, phiếu kiểm định chính `data/interim/human_audit_vi_120.csv` và phiếu cấp cây `data/interim/human_audit_vn_30_trees.csv` (tuyệt đối không kèm đáp án máy).
2. `sbifagat_member1_vn_sealed_machine_keys.zip` — gói niêm phong riêng chứa `data/interim/pilot_calibration_vi_20_machine_key.csv`, `data/interim/human_audit_vi_120_machine_key.csv` và `data/interim/human_audit_vn_30_trees_machine_key.csv` (chỉ giải nén đối chiếu `machine_vs_gold` sau khi 2 bạn chấm xong phiếu mù).

*Lưu ý:* Ở chế độ mặc định công bằng không chạy `auto_loop` (nếu muốn xuất thêm chẩn đoán Validation `outputs/auto_loop_vn/{loop_diagnostics,worst_val_cases_for_review}.json`, thêm cờ `--run-auto-loop` ở Bước 4).